# Copy `.tif` files from folders created on 28-Feb or 1-Mar

This notebook:

1. Takes a **source folder** path as input.
2. Finds all **subfolders directly inside** that source folder whose creation date is **28-Feb** or **1-Mar**.
3. Recursively walks through each matching folder until the deepest levels and collects all `.tif` files.
4. Copies those `.tif` files into a **destination folder**.

> Notes:
> - On some systems, folder “creation date” may not be truly available. In that case, the script falls back to the folder metadata change time exposed by Python.
> - If two files have the same name, the notebook automatically renames the later one to avoid overwriting.


In [5]:

from pathlib import Path
from datetime import datetime
import os
import shutil

# ===== USER INPUTS =====
source_folder = r"D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025"
destination_folder = r"D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\orders_all_delta4"

# Target month-day pairs
TARGET_DATES = {(5, 11), (5, 12), (5, 13), (5, 14)}  # (month, day)


In [2]:

def get_creation_datetime(path: Path) -> datetime:
    """
    Return the folder creation datetime when available.
    Falls back to metadata change time if birth time is unavailable.
    """
    stat = path.stat()

    # Windows usually has st_ctime as creation time.
    # Some Unix/macOS systems may expose st_birthtime.
    if hasattr(stat, "st_birthtime"):
        ts = stat.st_birthtime
    else:
        ts = stat.st_ctime

    return datetime.fromtimestamp(ts)


def matches_target_date(path: Path, target_dates=TARGET_DATES) -> bool:
    dt = get_creation_datetime(path)
    return (dt.month, dt.day) in target_dates


def unique_destination_path(dest_dir: Path, filename: str) -> Path:
    """
    Prevent overwriting if files with same name already exist.
    """
    candidate = dest_dir / filename
    if not candidate.exists():
        return candidate

    stem = candidate.stem
    suffix = candidate.suffix
    counter = 1
    while True:
        new_candidate = dest_dir / f"{stem}_{counter}{suffix}"
        if not new_candidate.exists():
            return new_candidate
        counter += 1


In [3]:

def collect_matching_top_level_folders(source_dir: Path):
    """
    Find folders directly inside source_dir whose creation date is 28-Feb or 1-Mar.
    """
    if not source_dir.exists():
        raise FileNotFoundError(f"Source folder does not exist: {source_dir}")
    if not source_dir.is_dir():
        raise NotADirectoryError(f"Source path is not a directory: {source_dir}")

    matched = []
    for item in source_dir.iterdir():
        if item.is_dir():
            try:
                if matches_target_date(item):
                    matched.append(item)
            except Exception as e:
                print(f"Skipping {item} due to error reading timestamps: {e}")
    return matched


def copy_tif_files_from_folders(folders, destination_dir: Path):
    """
    Recursively walk through each folder and copy all .tif/.tiff files to destination_dir.
    """
    destination_dir.mkdir(parents=True, exist_ok=True)

    copied_files = []
    for folder in folders:
        for tif_path in folder.rglob("*"):
            if tif_path.is_file() and tif_path.suffix.lower() in {".tif", ".tiff"}:
                dest_path = unique_destination_path(destination_dir, tif_path.name)
                shutil.copy2(tif_path, dest_path)
                copied_files.append((str(tif_path), str(dest_path)))

    return copied_files


In [6]:

# Convert to Path objects
source_dir = Path(source_folder)
destination_dir = Path(destination_folder)

# Step 1: Find matching folders
matched_folders = collect_matching_top_level_folders(source_dir)

print(f"Matched top-level folders: {len(matched_folders)}")
for folder in matched_folders:
    print(" -", folder, "| created:", get_creation_datetime(folder))

# Step 2 + 3: Recursively copy .tif files from those folders
copied = copy_tif_files_from_folders(matched_folders, destination_dir)

print(f"\nTotal .tif/.tiff files copied: {len(copied)}")


Matched top-level folders: 30
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi32 | created: 2026-05-11 14:49:27.713803
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi33 | created: 2026-05-11 21:14:22.542204
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi34 | created: 2026-05-11 21:19:02.425071
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi35 | created: 2026-05-11 21:21:39.474191
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi36 | created: 2026-05-11 21:25:41.065095
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PSScene_rgb_2025\aoi37 | created: 2026-05-11 21:34:26.503162
 - D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\inference\planet_downloads_PS

In [7]:

# Optional: inspect copied file mappings
for src, dst in copied[:20]:
    print(f"FROM: {src}")
    print(f"TO:   {dst}")
    print("-" * 80)

if len(copied) > 20:
    print(f"... and {len(copied) - 20} more")


FROM: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\order_00_9612e10b-a83a-43b3-a74d-66e8f2fced9f\9612e10b-a83a-43b3-a74d-66e8f2fced9f\PSScene\20250927_061113_79_24db_3B_Visual_clip_reproject_file_format.tif
TO:   D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta3\20250927_061113_79_24db_3B_Visual_clip_reproject_file_format.tif
--------------------------------------------------------------------------------
FROM: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\order_01_5e13f4f8-1f2a-4955-b604-d6a538a1e5af\5e13f4f8-1f2a-4955-b604-d6a538a1e5af\PSScene\20250927_061115_96_24db_3B_Visual_clip_reproject_file_format.tif
TO:   D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta3\20250927_061115_96

In [ ]:

# Optional helper:
# If you want to search ALL nested subfolders under source_folder for folders
# that themselves were created on 28-Feb or 1-Mar, use this alternative function.

def collect_all_matching_folders_recursive(source_dir: Path):
    matched = []
    for item in source_dir.rglob("*"):
        if item.is_dir():
            try:
                if matches_target_date(item):
                    matched.append(item)
            except Exception as e:
                print(f"Skipping {item} due to error reading timestamps: {e}")
    return matched

# Example:
# matched_folders = collect_all_matching_folders_recursive(source_dir)
# copied = copy_tif_files_from_folders(matched_folders, destination_dir)
